# 05: Send your circuit to a quantum computer

You will build a Bell circuit, check its ideal result, choose a hardware target, and submit it through FlagQuantum.
QSteed compiles the circuit for the selected device. Quafu runs it and returns measurement counts.
A *shot* is one execution followed by a measurement; we will request 1024 shots.

## 1. Build the circuit yourself

These are the same operations you used in notebook 01.

In [ ]:
import flagquantum as fq

circuit = fq.Circuit(2)
circuit.h(0)
circuit.cx(0, 1)


## 2. Check the ideal answer before submitting

This local check gives us a reference for the hardware result.

In [ ]:
ideal_state = fq.run(circuit).to_statevector().reshape(-1)
ideal_probabilities = ideal_state.abs().square().tolist()
print("Ideal probabilities:", ideal_probabilities)


## 3. Choose the hardware and number of shots

Ask your instructor which backend is available today. The environment needs the QSteed plugin and a `QUAFU_API_TOKEN`.
Credentials belong in the platform's secret settings or process environment, never in notebook cells.
QSteed selects physical qubits using current calibration; we do not fix a historical mapping here.

In [ ]:
target = "quafu:Dongling"  # Confirm today's backend with your instructor.
shots = 1024
submit_to_hardware = False
print("Target:", target)
print("Shots:", shots)


## 4. Prepare a file for your result

This file records that a submission was started, even if the connection is later interrupted.
Use it to keep track of the task rather than submitting again without checking.

In [ ]:
from pathlib import Path

# Find the checkout whether Jupyter started in the repository or this folder.
ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "flagquantum").is_dir()
)
WORKSHOP = ROOT / "workshops/flagos2026"
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(exist_ok=True)
print("Your result folder:", OUTPUTS)


In [ ]:
import json
import os
from datetime import datetime, timezone

result_path = OUTPUTS / "quafu-bell.json"
hardware_result = None


## 5. Submit through the public API

Set `submit_to_hardware` to `True` above, then run this cell once.
The important call is `fq.run`: it compiles, submits, and waits for the hardware result.
The surrounding lines save the submission details and prevent reuse of an existing result file.

If you interrupt the wait or receive an error, the platform may already have accepted the task.
Keep the file and check the platform task list before submitting again.

In [ ]:
if submit_to_hardware:
    if not os.getenv("QUAFU_API_TOKEN"):
        raise RuntimeError("Ask your instructor to configure QUAFU_API_TOKEN.")
    if shots <= 0 or shots % 1024:
        raise ValueError("Use a positive multiple of 1024 shots.")
    record = {
        "source": "live",
        "target": target,
        "shots": shots,
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "status": "submission_started",
    }
    with result_path.open("x") as output:
        json.dump(record, output, indent=2)

    hardware_result = fq.run(
        circuit,
        compiler="qsteed",
        target=target,
        shots=shots,
        name="flagos2026_bell",
    )
    print("Task ID:", hardware_result.provenance["task_id"])
else:
    print("No hardware task submitted. Your circuit and settings are ready.")


## 6. Read and save the counts

A count tells you how many shots produced an outcome such as `00`.
Check that the counts add up to the number of shots requested. Notebook 06 will load this saved result.

In [ ]:
if hardware_result is not None:
    counts = hardware_result.counts[0]
    print("Measurement counts:", counts)
    assert sum(counts.values()) == shots
    record.update(
        status="completed",
        task_id=str(hardware_result.provenance["task_id"]),
        counts=counts,
    )
    result_path.write_text(json.dumps(record, indent=2))
    print("Saved result:", result_path)


## Think about the result

Are `01` and `10` completely absent, as in the ideal simulation? Why might they appear?
A task ID identifies the submission; a completed result with counts shows that it ran.

Read the [Quafu guide](../../../../docs/guides/QUAFU_BACKEND.md) to explore logical-to-physical qubit mapping.
The physical circuit returned by Quafu is evidence from execution, not a guarantee of the final circuit available before submission.